In [ ]:
%load_ext autoreload
%autoreload 2

import argparse
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import mplhep
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader as CnnDataLoader
from torch_geometric.loader import DataLoader

from analysis.utils.datasets import CNNProjectionDataset
from analysis.utils.geometry import Geometry, get_rebinned_geometry
from analysis.utils.models import ClassifierProjectionCNN, GraphClassifier
from analysis.utils.plot_utils import setup
from analysis.utils.torch_data_utils import evaluate_model as evaluate_cnn_model
from analysis.utils.torch_geometric_utils import evaluate_model
from analysis.utils.torch_utils import (
    get_device,
    load_weights,
    show_model,
    train,
)
from analysis.utils.truth import TruthParticle, get_truth_particle
from analysis.utils.units import um
from analysis.utils.utils import (
    get_figures_path,
    get_npy_path,
    get_parquet_path,
    get_torch_path,
    get_weights_path,
)
from analysis.utils.validation_utils import (
    get_accuracy_dict,
    plot_confusion_matrix,
    plot_precision_recall_curve,
    plot_probabilities,
    plot_roc_curve,
    plot_training_metrics,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
setup()

In [3]:
@dataclass
class NuType:
    idx: int
    label: str
    latex_name: str


nue = NuType(0, "nue", r"CC $\nu_{e}$")
num = NuType(1, "num", r"CC $\nu_{\mu}$")
nun = NuType(2, "nun", "NC")
nus = [nue, num, nun]

In [ ]:
device = get_device()
criterion = nn.CrossEntropyLoss()

Using device: cpu


# CNN

In [18]:
cnn_geometry_label = "./"
cnn_torch_path = get_torch_path() / cnn_geometry_label / "cnn.pt"
cnn_weights_path = get_weights_path() / cnn_geometry_label
cnn_figures_path = get_figures_path() / cnn_geometry_label

In [19]:
cnn_model = ClassifierProjectionCNN(feature_dim=128, num_classes=3)
cnn_model = cnn_model.to(device)
load_weights(cnn_model, device, cnn_weights_path / "model_weights.pth")

cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4)

In [24]:
cnn_dataset = torch.load(cnn_torch_path, weights_only=False, map_location="cpu")

_, _test_indices = train_test_split(
    range(len(cnn_dataset)), test_size=0.1, random_state=42, stratify=cnn_dataset.labels
)

cnn_test_dataset = torch.utils.data.Subset(cnn_dataset, _test_indices)
cnn_test_loader = CnnDataLoader(
    cnn_test_dataset, batch_size=64, shuffle=False, pin_memory=True
)

In [28]:
cnn_y_true, cnn_y_pred, cnn_y_prob = evaluate_cnn_model(
    model=cnn_model, test_loader=cnn_test_loader, device=device
)

Evaluating: 100%|██████████| 17/17 [00:02<00:00,  6.83it/s]


# GNN

In [5]:
graph_geometry_label = "gcn_200um_bins_10000_10003"
graph_torch_path = get_torch_path() / graph_geometry_label
graph_weights_path = get_weights_path() / graph_geometry_label
graph_figures_path = get_figures_path() / graph_geometry_label

In [ ]:
graph_model = GraphClassifier(num_node_features=4, hidden_channels=64, num_classes=3)
graph_model = graph_model.to(device)
load_weights(graph_model, device, graph_weights_path / "model_weights.pth")

graph_optimizer = torch.optim.Adam(graph_model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
graph_test_dataset = torch.load(graph_torch_path / "test_dataset.pt")
graph_test_loader = DataLoader(
    graph_test_dataset, batch_size=1, shuffle=False, drop_last=True
)

In [8]:
graph_y_true, graph_y_pred, graph_y_prob = evaluate_model(
    model=graph_model, test_loader=graph_test_loader, device=device
)

Evaluating: 100%|██████████| 297/297 [01:45<00:00,  2.81it/s]


# Comparison

In [30]:
cnn_acc_dict = get_accuracy_dict(
    y_true=cnn_y_true, y_pred=cnn_y_pred, class_names=[nu.label for nu in nus]
)

graph_acc_dict = get_accuracy_dict(
    y_true=graph_y_true, y_pred=graph_y_pred, class_names=[nu.label for nu in nus]
)

In [33]:
nus

[NuType(idx=0, label='nue', latex_name='CC $\\nu_{e}$'),
 NuType(idx=1, label='num', latex_name='CC $\\nu_{\\mu}$'),
 NuType(idx=2, label='nun', latex_name='NC')]

In [ ]:
pd.DataFrame(
    {
        nu.label: [f"{cnn_acc_dict[nu.label]:.2f}", graph_acc_dict[nu.label]]
        for nu in nus
    },
    index=["CNN", "GCN"],
)

,nue,num,nun
CNN,0.73,0.75,0.69
GCN,0.828283,0.584906,0.532609


In [ ]:
pd.DataFrame()